# 📦 Notebook 1: Inventory Management Across Micro-Fulfillment Centers

Gopuff doesn’t use third-party stores. It owns **hundreds of small warehouses** (micro-fulfillment centers, or DCs) spread across a city. When a customer opens the app, the system must instantly show what’s available — by combining inventory from every DC that can deliver to them.

This notebook covers:
1. How inventory is structured across multiple DCs
2. Querying aggregated availability for a customer location
3. Placing orders with **atomic transactions** (no double-booking)
4. Speeding up reads with **Redis cache-aside**

## Learning Goals

- Understand the difference between an **Item** (catalog entry) and **Inventory** (physical stock at a DC)
- Write SQL that aggregates inventory across multiple warehouses
- Use Postgres `SERIALIZABLE` transactions to prevent two customers from buying the same last item
- Implement cache-aside with Redis to meet the < 100 ms latency requirement

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/gopuff
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, DB `gopuff`
- **RedisInsight** (Redis GUI): http://localhost:5540
  Add database → Host `redis`, Port `6379`

### Kernel Selection (VS Code)
Select the `.venv` kernel from the kernel picker (top-right of the notebook).
If it doesn’t appear, reload the VS Code window (`Cmd+Shift+P` → “Reload Window”).

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "gopuff",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True,
}


def get_db_connection():
    """Open a new Postgres connection."""
    return psycopg2.connect(**DB_CONFIG)


def get_redis_client():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)


conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM distribution_centers")
dc_count = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM items")
item_count = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM inventory")
inv_count = cur.fetchone()[0]
conn.close()

r = get_redis_client()
r.ping()

print(f"✅ Connected! {dc_count} DCs, {item_count} items, {inv_count} inventory rows")
print(f"✅ Redis is up")

## 📋 Requirements & Back-of-the-Envelope

### Functional
1. Show a customer **what's available** at their address — aggregated across every DC that can reach them.
2. Let a customer **place an order**, decrementing stock atomically.
3. Never **oversell**: two customers must not both get the last unit.
4. Restock / inventory adjustments from warehouse staff.

**Out of scope**: payments, driver dispatch, returns, catalog management.

### Non-functional
| Requirement | Target | Why |
|---|---|---|
| Availability query | p99 < 100 ms | It's the app's home screen |
| Order consistency | **Strongly consistent** | Overselling costs a refund, an apology, and a customer |
| Availability freshness | Seconds of staleness OK | Browsing a slightly stale count is fine; *ordering* is not |
| Scale | 10k DCs, 100k SKUs, 10M orders/day | Given in the problem statement |

Note the split — the same data needs **weak** consistency on the read path and
**strong** consistency on the write path. Every design decision below falls out
of that one sentence.

In [ ]:
# ── Back-of-the-envelope ────────────────────────────────────────────────
DC_COUNT             = 10_000
SKUS_PER_DC          = 3_000       # a micro-fulfillment centre is small
ORDERS_PER_DAY       = 10_000_000
BROWSE_PER_ORDER     = 50          # app opens, category scrolls, searches
NEARBY_DCS           = 3           # DCs that can reach a given address
INVENTORY_ROW_BYTES  = 80
CACHE_ENTRY_BYTES    = 50
PEAK_MULTIPLIER      = 3
SEC_PER_DAY          = 86_400

orders_per_s   = ORDERS_PER_DAY / SEC_PER_DAY
browse_per_s   = orders_per_s * BROWSE_PER_ORDER

inventory_rows  = DC_COUNT * SKUS_PER_DC
inventory_bytes = inventory_rows * INVENTORY_ROW_BYTES

# Every uncached availability query aggregates the nearby DCs' whole catalogue.
rows_per_query   = NEARBY_DCS * SKUS_PER_DC
row_reads_per_s  = browse_per_s * PEAK_MULTIPLIER * rows_per_query

# Cache sized per-DC (see the warning below about caching per DC *set*).
cache_bytes = DC_COUNT * SKUS_PER_DC * CACHE_ENTRY_BYTES

print("📐 Back-of-the-envelope")
print("=" * 70)
print(f"  Orders:              {orders_per_s:>12,.0f} /s avg   "
      f"{orders_per_s * PEAK_MULTIPLIER:>10,.0f} /s peak")
print(f"  Availability reads:  {browse_per_s:>12,.0f} /s avg   "
      f"{browse_per_s * PEAK_MULTIPLIER:>10,.0f} /s peak")
print(f"  Read:write ratio     {BROWSE_PER_ORDER:>12,}:1")
print()
print(f"  Inventory rows:      {inventory_rows:>12,}")
print(f"  Inventory size:      {inventory_bytes / 1e9:>12,.1f} GB   "
      f"← the ENTIRE table fits in RAM")
print()
print(f"  Rows an uncached query touches: {rows_per_query:>10,}")
print(f"  Row reads/s at peak, uncached:  {row_reads_per_s / 1e6:>10,.0f} million/s  ❌")
print(f"  Redis working set (per-DC keys):{cache_bytes / 1e9:>10,.1f} GB  ✅")

### What those numbers decide

- **~116 orders/s but ~5,800 availability reads/s** (≈17k/s at peak). A 50:1
  read:write ratio. The write path is tiny — which is *why* we can afford to
  make it strictly serializable and slow.
- **~150 million row reads/s** if availability is served straight from
  Postgres. That is the number that forces a cache. Not latency — throughput.
- **The whole inventory table is ~2.4 GB.** That is the pleasant surprise in
  this problem: your entire source of truth fits in memory. You are not caching
  because the data is big; you are caching because the *query* (a GROUP BY over
  9,000 rows) is expensive to repeat 17,000 times a second.
- **~1.5 GB of Redis** to hold per-DC availability. Cheap.

> ⚠️ **A bug in the caching code you're about to write.** Later in this notebook
> we cache under the key `availability:1,2,6` — one entry per *set* of nearby
> DCs. With 10,000 DCs the number of distinct nearby-sets is enormous, so the
> hit rate collapses and each order invalidates an unknown number of keys.
> The fix is to cache **one key per DC** (`availability:dc:1`) and merge the 3
> lists in the application. Cost: 3 Redis round trips instead of 1, plus merge
> CPU. Benefit: a bounded 10,000-key keyspace and single-key invalidation.
> We keep the naive version below because it reads more clearly — but say the
> per-DC version out loud in an interview.

## 1️⃣ Understanding the Data Model

Think of it like a convenience store chain:

- **Item** = “Coca-Cola 12-pack” (the *type* of product in the catalog)
- **Inventory** = “DC Downtown has 45 Coca-Cola 12-packs” (the *physical count* at a location)
- **Distribution Center** = a small warehouse at a specific lat/lon

A single item can exist in many DCs. The customer sees the *total* across all nearby DCs.

In [ ]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT id, name, city, latitude, longitude, capacity_sqft
    FROM distribution_centers
    ORDER BY id
""")

print("🏭 Distribution Centers")
print("=" * 75)
for dc in cur.fetchall():
    print(f"  DC {dc['id']:2d} | {dc['name']:<18s} | {dc['city']:<12s} | "
          f"({dc['latitude']:.4f}, {dc['longitude']:.4f}) | {dc['capacity_sqft']} sqft")

conn.close()

In [ ]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT
        dc.name AS dc_name,
        i.name  AS item_name,
        inv.quantity,
        inv.reorder_point
    FROM inventory inv
    JOIN distribution_centers dc ON dc.id = inv.dc_id
    JOIN items i ON i.id = inv.item_id
    WHERE i.name = 'Coca-Cola 12-pack'
    ORDER BY inv.quantity DESC
""")

rows = cur.fetchall()
total = sum(r['quantity'] for r in rows)

print("🥤 Coca-Cola 12-pack — Inventory by DC")
print("=" * 50)
for r in rows:
    bar = "█" * (r['quantity'] // 2)
    print(f"  {r['dc_name']:<18s} | {r['quantity']:4d} units {bar}")
print(f"{'':>18s}   ------")
print(f"{'Total':>18s} | {total:4d} units")

conn.close()

## 2️⃣ Aggregated Availability Query

When a customer opens the app, we need to:
1. Find which DCs can deliver to their location (we’ll cover this in Notebook 2)
2. Sum up inventory across those DCs for every item

For now, let’s assume we already know the nearby DC IDs and focus on the aggregation query.

**Key insight**: The customer doesn’t care *which* warehouse the item comes from. They just want to know “is Coke available and how many?”

In [ ]:
def get_availability(dc_ids):
    """
    Given a list of nearby DC IDs, return aggregated availability.
    Each item shows the total quantity across all the given DCs.
    """
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT
            i.id        AS item_id,
            i.name      AS item_name,
            i.category,
            i.base_price,
            SUM(inv.quantity) AS total_quantity
        FROM inventory inv
        JOIN items i ON i.id = inv.item_id
        WHERE inv.dc_id = ANY(%s)
          AND inv.quantity > 0
        GROUP BY i.id, i.name, i.category, i.base_price
        ORDER BY i.category, i.name
    """, (dc_ids,))

    results = cur.fetchall()
    conn.close()
    return results


nearby_dcs = [1, 2, 6]
availability = get_availability(nearby_dcs)

print(f"📋 Availability for DCs {nearby_dcs}")
print(f"   ({len(availability)} items in stock)\n")

current_cat = None
for item in availability:
    if item['category'] != current_cat:
        current_cat = item['category']
        print(f"\n  ── {current_cat} ──")
    print(f"    {item['item_name']:<28s} ${float(item['base_price']):>6.2f}  "
          f"({item['total_quantity']} in stock)")

## 3️⃣ The Naive Order Path (⚠️ Broken Under Concurrency)

Before we write the "right" version, let's write the version a junior engineer might write first — no transaction, just "read the quantity, subtract, write it back." This is **bad practice** on purpose, and we'll prove it breaks.

```
Thread A:           Thread B:
  read qty = 1
                      read qty = 1   ← both see stock!
  write qty = 0
                      write qty = 0  ← should be -1, we just oversold
```

The bug is called a **lost update** / **double-booking**. It happens any time two requests read the same row, compute something, and write back without coordination.


In [ ]:
# ⚠️ DO NOT USE IN PRODUCTION — this is the "bad" version on purpose.
import threading

def place_order_naive(customer_id, dc_id, item_id, quantity, barrier=None):
    """
    Naive order: read quantity, check, then decrement.
    No transaction. No row lock. Vulnerable to race conditions.
    Each call uses its own connection (same as the fixed version).
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Step 1: read current quantity
    cur.execute(
        "SELECT quantity FROM inventory WHERE dc_id = %s AND item_id = %s",
        (dc_id, item_id),
    )
    row = cur.fetchone()
    current_qty = row[0] if row else 0

    # Barrier forces both threads to reach here *before* either writes.
    # In the real world the interleaving is random — we just make it reproducible.
    if barrier is not None:
        barrier.wait()

    if current_qty < quantity:
        conn.close()
        return {"customer": customer_id, "success": False, "reason": "out of stock"}

    # Step 2: decrement (unconditionally — the bug!)
    cur.execute(
        "UPDATE inventory SET quantity = quantity - %s WHERE dc_id = %s AND item_id = %s",
        (quantity, dc_id, item_id),
    )
    conn.commit()
    conn.close()
    return {"customer": customer_id, "success": True, "took": current_qty}


# Reset the demo item to exactly 1 unit
conn = get_db_connection()
cur = conn.cursor()
cur.execute("UPDATE inventory SET quantity = 1 WHERE dc_id = 1 AND item_id = 17")
conn.commit()
conn.close()

# Two threads, one barrier, one item — classic oversell setup
barrier = threading.Barrier(2)
results = {}

def buyer(name, cid):
    results[name] = place_order_naive(cid, dc_id=1, item_id=17, quantity=1, barrier=barrier)

t1 = threading.Thread(target=buyer, args=("Alice", 100))
t2 = threading.Thread(target=buyer, args=("Bob", 200))
t1.start(); t2.start(); t1.join(); t2.join()

conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT quantity FROM inventory WHERE dc_id = 1 AND item_id = 17")
final_qty = cur.fetchone()[0]
conn.close()

for name, res in results.items():
    print(f"  {name}: {res}")
print(f"\n💥 Final inventory: {final_qty}  (negative = we sold something we didn't have)")
# Assert, don't just narrate. If this ever stops failing, the demo is lying.
assert final_qty == -1, f"expected the oversell, got quantity={final_qty}"
assert all(res["success"] for res in results.values()), results
print("   Both buyers were told 'yes' for one unit of stock. This is the bug.")


## 4️⃣ The Fix — Atomic Transactions

### The Double-Booking Problem

Imagine there’s **1 avocado pack left** at DC Downtown. Two customers tap “Buy” at the same time:

```
Customer A: reads quantity = 1  ✓ available!
Customer B: reads quantity = 1  ✓ available!
Customer A: decrements to 0, creates order
Customer B: decrements to -1 ← BUG! We promised something we don’t have
```

### The Fix: Postgres Transactions

We wrap the entire check-and-decrement in a **single database transaction** with `SERIALIZABLE` isolation. Postgres guarantees that if two transactions conflict, one of them will be rolled back.

Think of it like a lock on a fitting room door — only one person can try on the last pair of jeans at a time.

In [ ]:
def place_order(customer_id, dc_id, items_to_order,
                delivery_address, delivery_lat, delivery_lon):
    """
    Place an order atomically.
    items_to_order: [{"item_id": 1, "quantity": 2}, ...]

    Steps inside one transaction:
      1. Check inventory for every item
      2. If anything is out of stock -> roll back
      3. Decrement inventory
      4. Create the order + order items
      5. Commit
    """
    conn = get_db_connection()
    conn.set_session(isolation_level="SERIALIZABLE")
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        total_price = 0

        for entry in items_to_order:
            cur.execute("""
                SELECT inv.quantity, i.base_price
                FROM inventory inv
                JOIN items i ON i.id = inv.item_id
                WHERE inv.dc_id = %s AND inv.item_id = %s
                FOR UPDATE
            """, (dc_id, entry["item_id"]))

            row = cur.fetchone()
            if row is None or row["quantity"] < entry["quantity"]:
                conn.rollback()
                available = row["quantity"] if row else 0
                return {
                    "success": False,
                    "error": f"Item {entry['item_id']} — requested {entry['quantity']}, "
                             f"only {available} available at DC {dc_id}"
                }
            total_price += float(row["base_price"]) * entry["quantity"]

        for entry in items_to_order:
            cur.execute("""
                UPDATE inventory
                SET quantity = quantity - %s, updated_at = NOW()
                WHERE dc_id = %s AND item_id = %s
            """, (entry["quantity"], dc_id, entry["item_id"]))

        cur.execute("""
            INSERT INTO orders (customer_id, dc_id, status, total_price,
                                delivery_address, delivery_lat, delivery_lon)
            VALUES (%s, %s, 'confirmed', %s, %s, %s, %s)
            RETURNING id
        """, (customer_id, dc_id, total_price, delivery_address,
              delivery_lat, delivery_lon))
        order_id = cur.fetchone()["id"]

        for entry in items_to_order:
            cur.execute("SELECT base_price FROM items WHERE id = %s", (entry["item_id"],))
            price = cur.fetchone()["base_price"]
            cur.execute("""
                INSERT INTO order_items (order_id, item_id, quantity, unit_price)
                VALUES (%s, %s, %s, %s)
            """, (order_id, entry["item_id"], entry["quantity"], price))

        conn.commit()
        return {"success": True, "order_id": order_id, "total_price": total_price}

    except psycopg2.errors.SerializationFailure:
        conn.rollback()
        return {"success": False, "error": "Transaction conflict — please retry"}
    finally:
        conn.close()


result = place_order(
    customer_id=42, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 2}, {"item_id": 9, "quantity": 1}],
    delivery_address="123 Main St, Austin, TX",
    delivery_lat=30.2672, delivery_lon=-97.7431,
)

if result["success"]:
    print(f"✅ Order #{result['order_id']} placed! Total: ${result['total_price']:.2f}")
else:
    print(f"❌ Order failed: {result['error']}")

### 🧪 Simulating the Double-Booking Race Condition

Let’s prove the transaction works. We’ll set an item to quantity = 1, then try to buy it from two “customers” at the same time using threads.

In [ ]:
import threading

def run_race(round_no):
    """One customer's-worth of contention: 2 buyers, 1 unit of stock."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("UPDATE inventory SET quantity = 1 WHERE dc_id = 1 AND item_id = 17")
    conn.commit()
    conn.close()

    results = {}
    start = threading.Barrier(2)

    def try_order(customer_name, customer_id):
        start.wait()  # release both buyers at the same instant
        results[customer_name] = place_order(
            customer_id=customer_id, dc_id=1,
            items_to_order=[{"item_id": 17, "quantity": 1}],
            delivery_address=f"{customer_name}'s house",
            delivery_lat=30.27, delivery_lon=-97.74,
        )

    threads = [threading.Thread(target=try_order, args=(n, cid))
               for n, cid in (("Alice", 100), ("Bob", 200))]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT quantity FROM inventory WHERE dc_id = 1 AND item_id = 17")
    final = cur.fetchone()[0]
    conn.close()
    return results, final


print("🏁 Serializable order path — 5 rounds, 2 buyers, 1 unit each round")
print("=" * 72)
for round_no in range(1, 6):
    results, final = run_race(round_no)
    winners = [n for n, r in results.items() if r["success"]]
    losers = [f"{n}: {r['error']}" for n, r in results.items() if not r["success"]]
    print(f"  round {round_no}: stock_after={final:>2}   won={winners}   "
          f"rejected={losers}")

    # The two invariants that actually matter. A concurrency demo that
    # only works once has proven nothing, so we check every round.
    assert len(winners) == 1, f"expected exactly one winner, got {results}"
    assert final == 0, f"stock must never go negative or leak; got {final}"

print()
print("✅ 5/5 rounds: exactly one buyer succeeded and stock landed on 0, never -1.")
print()
print("What it cost us: the loser paid for a blocked row lock and, under")
print("SERIALIZABLE, may get a retryable conflict rather than a clean")
print("'out of stock'. That is why the next section adds a retry loop —")
print("serializable isolation moves work from 'silently wrong' to 'visibly")
print("contended', and the contention is yours to handle.")

## 5️⃣ Speeding Up Reads with Redis (Cache-Aside)

The interview requires availability queries under **100 ms**. Our estimate above put peak availability traffic at **~17k queries/second**, each aggregating ~9,000 inventory rows. Hitting Postgres every time is roughly 150 million row-reads per second — not happening.

### Cache-Aside Pattern

```
Customer Request
       │
       ▼
  ┌─ Check Redis ─┐
  │               │
  │  HIT?  ───── Return cached result (< 5 ms)
  │               │
  │  MISS? ───── Query Postgres
  │               │
  │               ├── Store in Redis (TTL = 60s)
  │               │
  │               └── Return result
  └───────────────┘
```

**Why a 60-second TTL?** Inventory changes when orders are placed, but a small delay is acceptable for *browsing*. When someone actually places an order, we check the real database (strong consistency for writes, eventual consistency for reads).

In [ ]:
CACHE_TTL_SECONDS = 60

def get_availability_cached(dc_ids):
    """
    Cache-aside availability lookup.
    Returns (items, source) where source is 'cache' or 'database'.
    """
    r = get_redis_client()
    cache_key = "availability:" + ",".join(str(d) for d in sorted(dc_ids))

    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), "cache"

    items = get_availability(dc_ids)
    serializable = [
        {k: (float(v) if hasattr(v, 'as_tuple') else v) for k, v in row.items()}
        for row in items
    ]
    r.setex(cache_key, CACHE_TTL_SECONDS, json.dumps(serializable))
    return serializable, "database"


r = get_redis_client()
r.flushdb()

start = time.time()
items, source = get_availability_cached([1, 2, 6])
first_ms = (time.time() - start) * 1000
print(f"1st call: {source:>8s} | {first_ms:.1f} ms | {len(items)} items")

start = time.time()
items, source = get_availability_cached([1, 2, 6])
second_ms = (time.time() - start) * 1000
print(f"2nd call: {source:>8s} | {second_ms:.1f} ms | {len(items)} items")

print(f"\n⚡ Cache speedup: {first_ms / max(second_ms, 0.01):.1f}x faster")

### Cache Invalidation on Order

When an order is placed, the cached availability is **stale** (it still shows the old quantity). We should invalidate the relevant cache entries so the next read fetches fresh data.

In [ ]:
def invalidate_availability_cache(dc_id):
    r = get_redis_client()
    pattern = "availability:*"
    deleted = 0
    for key in r.scan_iter(pattern):
        dc_ids_in_key = key.split(":")[1].split(",")
        if str(dc_id) in dc_ids_in_key:
            r.delete(key)
            deleted += 1
    return deleted


items, source = get_availability_cached([1, 2, 6])
print(f"Before order: source = {source}")

result = place_order(
    customer_id=99, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 1}],
    delivery_address="456 Oak Ave", delivery_lat=30.27, delivery_lon=-97.74,
)
print(f"Order result: {'✅' if result['success'] else '❌'}")

deleted = invalidate_availability_cache(dc_id=1)
print(f"Invalidated {deleted} cache key(s)")

items, source = get_availability_cached([1, 2, 6])
print(f"After invalidation: source = {source}  (fresh data!)")

## 📊 Performance Comparison

Let’s measure many reads with and without caching to see the difference at scale.

In [ ]:
import statistics

def benchmark(fn, label, iterations=50):
    times = []
    for _ in range(iterations):
        start = time.time()
        fn()
        times.append((time.time() - start) * 1000)

    print(f"\n📈 {label} ({iterations} iterations)")
    print(f"   Avg: {statistics.mean(times):.1f} ms")
    print(f"   Med: {statistics.median(times):.1f} ms")
    print(f"   P95: {sorted(times)[int(iterations * 0.95)]:.1f} ms")
    print(f"   Min: {min(times):.1f} ms | Max: {max(times):.1f} ms")
    return times

dc_ids = [1, 2, 6]
db_times = benchmark(lambda: get_availability(dc_ids), "Direct Postgres")

r = get_redis_client()
r.flushdb()
get_availability_cached(dc_ids)
cache_times = benchmark(lambda: get_availability_cached(dc_ids), "Redis Cache-Aside")

print(f"\n⚡ Average speedup: {statistics.mean(db_times) / statistics.mean(cache_times):.1f}x")

## 6️⃣ Retrying on Serialization Failures

`SERIALIZABLE` isolation isn't free — when two transactions conflict, Postgres **aborts one of them** with a `SerializationFailure`. In a real service you want to automatically retry instead of failing the user request.

This is the standard pattern used by any system that leans on serializable transactions (Postgres, CockroachDB, Spanner…): wrap the call in a small retry loop with backoff.


In [ ]:
import random

def place_order_with_retry(max_retries=3, **kwargs):
    """
    Retry place_order on serialization failures with exponential backoff + jitter.
    Real-world services usually cap retries at 3–5 and give up with a 503 after that.
    """
    for attempt in range(max_retries):
        result = place_order(**kwargs)
        # place_order already swallows SerializationFailure and returns an error dict,
        # so we check the error message to decide whether to retry.
        if result.get("success"):
            return {**result, "attempts": attempt + 1}
        if "conflict" not in result.get("error", "").lower():
            # Business failure (e.g. out of stock) — don't retry
            return {**result, "attempts": attempt + 1}
        # Exponential backoff with jitter: 20ms, 40ms, 80ms, …
        sleep_ms = (2 ** attempt) * 20 + random.randint(0, 10)
        time.sleep(sleep_ms / 1000)
    return {"success": False, "error": "max retries exceeded", "attempts": max_retries}


# Quick sanity run
result = place_order_with_retry(
    max_retries=3,
    customer_id=7, dc_id=1,
    items_to_order=[{"item_id": 1, "quantity": 1}],
    delivery_address="1 Retry Lane",
    delivery_lat=30.27, delivery_lon=-97.74,
)
print(result)


### 📝 Real-World Note: Inventory Reservations

In production, Gopuff-style services don't decrement stock at "Place Order." They **reserve** it when the customer enters checkout (e.g., hold for 5 minutes), then commit the decrement at payment success or release it on timeout / cancel.

The schema extension is typically:

```sql
CREATE TABLE inventory_holds (
    id SERIAL PRIMARY KEY,
    dc_id INT, item_id INT, quantity INT,
    customer_id INT,
    expires_at TIMESTAMP   -- hold auto-releases if payment stalls
);
```

The availability query then becomes `quantity - SUM(active_holds)`. We stop here in this notebook, but interviewers love this detail.


## 🔑 Key Takeaways

| Concept | What We Did |
|---------|-------------|
| **Item vs Inventory** | Items are catalog entries; Inventory is physical stock at a DC |
| **Aggregated availability** | `SUM(quantity)` across nearby DCs — customer sees one number |
| **Atomic ordering** | `SERIALIZABLE` transaction + `FOR UPDATE` row lock prevents double-booking |
| **Cache-aside** | Check Redis first → miss → query Postgres → store in Redis with TTL |
| **Cache invalidation** | Delete stale cache keys after an order changes inventory |

### Interview Tip

When asked about Gopuff, start with the **data model** (Item vs Inventory distinction), then explain **read path** (cache-aside for browsing) vs **write path** (serializable transactions for ordering). This shows you understand the different consistency requirements for reads and writes.

## 🧹 Cleanup

In [ ]:
r = get_redis_client()
keys = r.keys("availability:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")